# Imagine AI — SD 1.5 LCM-LoRA speed fine-tune

This notebook trains a few-step consistency LoRA using the project's 10,000 COCO training pairs. In Colab, select a T4 GPU (16 GB) or another GPU with at least 14 GiB available. Upload `coco_train.tar` and `coco_train.manifest.json` from `training_data/coco_sd15_lenbalanced/wds/` into `MyDrive/ImagineAI_LCM/` first. Training can take a while; checkpoints are saved to Drive so the run can resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{gpu_name}: {gpu_vram_gib:.1f} GiB VRAM')
if gpu_vram_gib < 14:
    raise RuntimeError('Use a GPU with about 16 GiB VRAM for this recipe.')

DRIVE_DIR = Path('/content/drive/MyDrive/ImagineAI_LCM')
ARCHIVE_DRIVE = DRIVE_DIR / 'coco_train.tar'
MANIFEST_DRIVE = DRIVE_DIR / 'coco_train.manifest.json'
assert ARCHIVE_DRIVE.is_file() and MANIFEST_DRIVE.is_file(), 'Upload the archive and manifest to MyDrive/ImagineAI_LCM.'

In [ ]:
%pip install -q "diffusers==0.35.1" "transformers<5" "peft>=0.14" "accelerate>=0.30" "webdataset==1.0.2" "braceexpand==0.1.7" safetensors

In [ ]:
import hashlib, json, shutil, urllib.request

meta = json.loads(MANIFEST_DRIVE.read_text())
h = hashlib.sha256()
with ARCHIVE_DRIVE.open('rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == meta['archive_sha256'], 'Archive hash mismatch.'
assert meta['training_pairs'] == 10000 and not meta['contains_validation_split']
ARCHIVE_LOCAL = Path('/content/coco_train.tar')
shutil.copy2(ARCHIVE_DRIVE, ARCHIVE_LOCAL)
print(f"Verified {meta['training_pairs']} training pairs; copied {ARCHIVE_LOCAL.stat().st_size:,} bytes.")

TRAINER_URL = 'https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/consistency_distillation/train_lcm_distill_lora_sd_wds.py'
TRAINER_SHA256 = '91574a77355539f44cacb229596eeb97e59da20ab969b2fe868132aaeb1047ec'
TRAINER = Path('/content/train_lcm_distill_lora_sd_wds.py')
urllib.request.urlretrieve(TRAINER_URL, TRAINER)
assert hashlib.sha256(TRAINER.read_bytes()).hexdigest() == TRAINER_SHA256, 'Trainer source hash mismatch.'
print('Pinned Hugging Face trainer verified.')

In [ ]:
import os, subprocess
OUTPUT_DIR = DRIVE_DIR / 'lcm_lora_coco_sd15_rank4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'

def run_lcm_training(total_steps):
    cmd = [
        'accelerate', 'launch', '--mixed_precision=fp16', str(TRAINER),
        '--pretrained_teacher_model=stable-diffusion-v1-5/stable-diffusion-v1-5',
        '--teacher_revision=451f4fe16113bff5a5d2269ed5ad43b0592e9a14',
        f'--output_dir={OUTPUT_DIR}', f'--train_shards_path_or_url={ARCHIVE_LOCAL}',
        '--resolution=512', '--train_batch_size=1', '--gradient_accumulation_steps=4',
        f'--max_train_steps={total_steps}', '--max_train_samples=10000',
        '--learning_rate=1e-4', '--lr_warmup_steps=50',
        '--lora_rank=4', '--lora_alpha=4', '--mixed_precision=fp16', '--cast_teacher_unet',
        '--gradient_checkpointing', '--vae_encode_batch_size=1', '--dataloader_num_workers=1',
        '--checkpointing_steps=250', '--checkpoints_total_limit=2', '--resume_from_checkpoint=latest',
        '--validation_steps=2500', '--report_to=none', '--seed=2026',
    ]
    subprocess.run(cmd, check=True, env=env)

print('Pilot: 250 total optimizer updates. Output:', OUTPUT_DIR)
run_lcm_training(250)

## Continue after the pilot

Record the pilot time and confirm it fits. If it is acceptable, run this cell to continue from checkpoint-250 to 2,500 total updates. If Colab ends a later session, rerun this cell to resume from the latest checkpoint.
```python
print('Continuing from checkpoint-250 to 2,500 total optimizer updates.')
run_lcm_training(2500)
```

## After training

The LoRA weights are in `MyDrive/ImagineAI_LCM/lcm_lora_coco_sd15_rank4/unet_lora/`. Keep them as an experimental artifact. Benchmark 4-, 8-, and 30-step generation with the same prompts and seeds, then review quality and failure rates. Do not replace the approved adapter or enable this output in the website until the comparisons pass. A T4 improves VRAM headroom; it does not guarantee a faster training step or a better image. Colab availability and runtime limits vary.